In [1]:
!nvidia-smi

Fri Aug 14 04:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.3 MB/s eta 0:00:00


In [3]:
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 43.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [4]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


### **LOAD THE IMPORT LIBRARIES**

In [5]:
from transformers import pipeline, set_seed, AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import matplotlib.pyplot as plt
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

#### **CHECK IF THE DEVICE IS USING CUDA**

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

#### **MODEL CHECKPOINT +  TOKENIZER**

In [7]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

#### **CREATE A MODEL**

In [8]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.28GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

#### **TAKE AND LOAD THE DATASET**

In [9]:
dataset_samsum = load_dataset('knkarthick/samsum')
dataset_samsum

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [10]:
dataset_samsum['train']['dialogue'][1]

'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great'

In [11]:
dataset_samsum['train'][1]['summary']

'Olivia and Olivier are voting for liberals in this election. '

In [12]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]
print("Split_length : ", split_lengths)
print('Features :', dataset_samsum['train'].column_names)
print("\n Dialogue :")

print(dataset_samsum['test'][1]['dialogue'])
print("\n Summary")
print(dataset_samsum['test'][1]['summary'])

Split_length :  [14731, 818, 819]
Features : ['id', 'dialogue', 'summary']

 Dialogue :
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

 Summary
Eric and Rob are going to watch a stand-up on youtube.


In [13]:
# PREPROCESS AND CONVERT THE DATA INTO VECTOR REPRESENTATION
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

    # Use text_target instead of the deprecated as_target_tokenizer context manager
    tokenizer_encodings = tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': tokenizer_encodings['input_ids']
    }

In [14]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [15]:
dataset_samsum_pt['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [16]:
dataset_samsum_pt['train']['input_ids'][1]

[18038,
 151,
 2632,
 127,
 119,
 6228,
 118,
 115,
 136,
 2974,
 152,
 10463,
 151,
 35884,
 130,
 329,
 107,
 18038,
 151,
 2587,
 314,
 1242,
 10463,
 151,
 1509,
 1]

In [17]:
dataset_samsum_pt['train']['attention_mask'][1]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [18]:
dataset_samsum_pt['train']['labels'][1]

[18038, 111, 34296, 127, 6228, 118, 33195, 115, 136, 2974, 107, 110, 1]

#### **TRAINING**

In [19]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_coll = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [20]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs=1, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps = 10,
    eval_strategy='steps', eval_steps=106, save_steps=1e6,
    gradient_accumulation_steps=16
)

In [21]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_coll,
    train_dataset=dataset_samsum_pt['test'],  # Using 'test' dataset for quick testing
    eval_dataset=dataset_samsum_pt['validation']
)

# Enable gradient checkpointing to save memory
model_pegasus.gradient_checkpointing_enable()

torch.cuda.empty_cache()
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
52,47.229971,2.415025


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=48.390545478233925, metrics={'train_runtime': 439.373, 'train_samples_per_second': 1.864, 'train_steps_per_second': 0.118, 'total_flos': 314203859361792.0, 'train_loss': 48.390545478233925, 'epoch': 1.0})

#### **EVALUATION**

In [24]:
#THIS EVALUATION CODE WILL BE ACUTOMATICALLY GETTING IN THE HUGGING FACE WE UNDER THE MODEL DESCRIPTION

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the data into small batches that can be processed simultaneously."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]


def calculate_metrics_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="dialogue",
                               column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                          padding="max_length", return_tensors="pt")

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8, num_beams=8, max_length=128
        )

        # Decode the generated texts
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)
                             for s in summaries]

        decoded_summaries = [d.replace("", "") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Compute and return the final score
    score = metric.compute()
    return score

In [26]:
rouge_metric = evaluate.load("rouge")

scores = calculate_metrics_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size=2  #WE USE HERE [0:10] A SMALL DATA JUTS FOR FAST PRATICE
)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
df = pd.DataFrame(scores, index=['pegasus'])
df

100%|██████████| 5/5 [00:31<00:00,  6.22s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.324607,0.094385,0.249758,0.257658


In [27]:
#SAVE THE MODEL
#SAVE THE TOKENIZER
model_pegasus.save_pretrained('pegasus-samsum-model')
tokenizer.save_pretrained('tokenizer')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

### **HERE WE USE AND LOAD OUR MODEL**

In [29]:
#LOAD
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [31]:
dataset_samsum['test'][0]['dialogue']

"Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye"

In [43]:
# PREDICTION
import torch
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. Aggressively clear memory to avoid OOM (Out Of Memory) errors
# We delete any variables that might be holding onto GPU memory
if 'model_pegasus' in globals():
    del model_pegasus
if 'model_for_pipeline' in globals():
    del model_for_pipeline
if 'trainer' in globals():
    del trainer

gc.collect()
torch.cuda.empty_cache()

# 2. Setup generation arguments and test data
gen_krwgs = {'length_penalty': 0.8, "num_beams": 8, 'max_length': 128}
sample_text = dataset_samsum['test'][0]['dialogue']
reference = dataset_samsum['test'][0]['summary']

# 3. Load the fine-tuned model
# We use the local path where we saved the model earlier
model_for_prediction = AutoModelForSeq2SeqLM.from_pretrained('pegasus-samsum-model').to(device)

# 4. Tokenize the input text
inputs = tokenizer(sample_text, max_length=1024, truncation=True, return_tensors="pt")

print('Dialogue')
print(sample_text)
print('\nReference')
print(reference)
print('\nModel Summary:')

# 5. Generate summary using the model's generate method directly
with torch.no_grad():
    summaries = model_for_prediction.generate(
        input_ids=inputs["input_ids"].to(device),
        attention_mask=inputs["attention_mask"].to(device),
        **gen_krwgs
    )

# 6. Decode and clean the special newline tokens
decoded_summary = tokenizer.decode(summaries[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
clean_summary = decoded_summary.replace("<n>", " ")

print(clean_summary)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Dialogue
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda: Ask Larry Amanda: He called her last time we were at the park together. Hannah: I'd rather you texted him. Amanda: Just text him.
